# 1. The data

The public data are one `npz` file per sample, with a `meta` entry (JSON) that names the source
record and how the arrays were made. Set `DATA_ROOT` to the folder you downloaded.

| folder | content |
|---|---|
| `frames/{train,val}` | HMI 45 s frames, central 512 px, float16 (the values the training reads) |
| `frames/test` | the test frames, float32 |
| `stacks/<split>` | one hourly group: the input frame and its 21-frame stack |
| `pairs/mdi` | HMI-MDI pairs on the MDI grid |
| `pairs/phi` | HMI-PHI pairs on the HMI and the HRT grids |
| `manifests` | which files form each split and group |

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")          # the repository root, one level up from notebooks/
DATA_ROOT = Path("../data")       # the public data: manifests/, frames/, stacks/, pairs/, checkpoints/, results/
import csv, json
import numpy as np
import matplotlib.pyplot as plt
import pipeline

## Manifests: the splits and their size

In [ ]:
for name in ("frames.csv", "stacks.csv"):
    counts = {}
    for r in csv.DictReader(open(DATA_ROOT / "manifests" / name)):
        counts[r["split"]] = counts.get(r["split"], 0) + 1
    print(name, counts)

## A training frame (float16, gauss)

In [ ]:
group = pipeline.frame_groups(DATA_ROOT, "train")[0]
frame = pipeline.load_frame(group[0])
with np.load(group[0]) as z:
    print(json.dumps(json.loads(str(z["meta"])), indent=1))
plt.imshow(frame.astype(np.float32), cmap="gray", vmin=-30, vmax=30, origin="lower")
plt.colorbar(label="G"); plt.title(Path(group[0]).name); plt.show()

## A test group: the input frame and its 21-frame stack

In [ ]:
f = pipeline.stack_files(DATA_ROOT, "test")[0]
x, y = pipeline.load_stack(f)
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, im, t in zip(ax, (x, y, x - y), ("input frame", "21-frame stack", "input - stack")):
    a.imshow(im, cmap="gray", vmin=-30, vmax=30, origin="lower"); a.set_title(t); a.axis("off")
plt.show()
print("noise of the single frame (robust sigma of input - stack): %.2f G" % pipeline.robust_sigma(x - y))

## An HMI-MDI pair on the MDI grid

In [ ]:
f = sorted((DATA_ROOT / "pairs" / "mdi").glob("*.npz"))[0]
with np.load(f) as z:
    hmi, mdi, disk = z["hmi"], z["mdi"], z["disk"]
    print(json.dumps(json.loads(str(z["meta"])), indent=1))
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, im, t in zip(ax, (hmi, mdi, disk), ("HMI on the MDI grid", "MDI", "r < 0.85 R_sun")):
    a.imshow(im, cmap="gray", vmin=-30 if im.dtype != bool else 0, vmax=30 if im.dtype != bool else 1, origin="lower")
    a.set_title(t); a.axis("off")
plt.show()

## An HMI-PHI epoch

`hmi` and `phi_on_hmi` are on the HMI grid; `phi` and `hmi_on_hrt` on the HRT grid (PHI
co-aligned to HMI). `coords_hrt` holds, for every HRT pixel, its position in `hmi_foot`, so a
denoised HMI crop can be put on the HRT grid with `pipeline.bilinear` (Table 2c).

In [ ]:
f = sorted((DATA_ROOT / "pairs" / "phi").glob("*.npz"))[0]
z = dict(np.load(f))
meta = json.loads(str(z.pop("meta")))
print({k: meta[k] for k in ("phi_file", "hmi_record", "offset_s", "sep_deg", "d_au", "shift_arcsec", "corr_after")})
fig, ax = plt.subplots(1, 4, figsize=(17, 4))
for a, k in zip(ax, ("hmi", "phi_on_hmi", "phi", "hmi_on_hrt")):
    a.imshow(z[k], cmap="gray", vmin=-30, vmax=30, origin="lower"); a.set_title(k); a.axis("off")
plt.show()
print("HMI crop on the HRT grid through the stored coordinates:", pipeline.bilinear(z["hmi_foot"], z["coords_hrt"]).shape)